# Versao 12 - Visao Geral Dos Dados

A `versao12` nasce como uma sintese das melhores ideias das versoes `7`, `9`, `10` e `11`. A pergunta agora deixa de ser se cada bloco arquitetural funciona isoladamente e passa a ser: o que acontece quando reunimos profundidade alta, hierarquia temporal, informacao tabular, mascaras operacionais, contexto de fonte e multitarefa em uma unica rede?

## Hipotese central

A `versao12` parte da seguinte hipotese metodologica:

- a profundidade extra da `versao7` ajuda, desde que nao venha sozinha;
- a hierarquia temporal da `versao9` ajuda a organizar a serie em escalas locais e globais;
- as mascaras, o `source_id` e a multitarefa da `versao10` ajudam a tornar a rede mais fiel ao `3W`;
- a limpeza de features vazias da `versao11` evita carregar ruido estrutural desnecessario.

Por isso, a `versao12` combina:

- ramos separados para variaveis continuas e de estado;
- janelas temporais com codificadores locais profundos;
- pooling composto com `last hidden + mean + attention`;
- codificador de contexto entre janelas;
- fusao com `X_tab`;
- mascaras `missing` e `frozen`;
- embedding de contexto por tipo de fonte;
- cabeca principal global e cabecas auxiliares por observacao.


In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao12" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from versao12.pipeline_v12 import (
    SELECTED_CONTINUOUS_SENSOR_COLUMNS,
    SELECTED_FEATURE_COLUMNS,
    SELECTED_STATE_SENSOR_COLUMNS,
    SOURCE_TYPE_MAPPING,
    build_feature_selection_report,
    discover_series_manifest,
    load_attribute_catalog,
    load_event_catalog,
)

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
manifest = discover_series_manifest(DATASET_ROOT)
attribute_catalog = load_attribute_catalog(DATASET_ROOT)
event_catalog = load_event_catalog(DATASET_ROOT)
feature_report = build_feature_selection_report()

print("Numero total de series:", len(manifest))
print("Numero de classes globais:", manifest["class_label"].nunique())
print("Numero de fontes:", manifest["source_type"].nunique())
print("Features mantidas:", len(SELECTED_FEATURE_COLUMNS))
print("Features continuas:", len(SELECTED_CONTINUOUS_SENSOR_COLUMNS))
print("Features de estado:", len(SELECTED_STATE_SENSOR_COLUMNS))
print("Mapeamento de origem:", SOURCE_TYPE_MAPPING)


Numero total de series: 2228
Numero de classes globais: 10
Numero de fontes: 3
Features mantidas: 18
Features continuas: 9
Features de estado: 9
Mapeamento de origem: {'well': 0, 'simulated': 1, 'drawn': 2}


In [2]:
resumo = pd.DataFrame(
    [
        {
            "n_series": len(manifest),
            "n_classes_globais": manifest["class_label"].nunique(),
            "n_sources": manifest["source_type"].nunique(),
            "n_features_originais": len(feature_report),
            "n_features_mantidas": int(feature_report["selected_for_modeling"].sum()),
            "n_features_removidas": int((~feature_report["selected_for_modeling"]).sum()),
        }
    ]
)
display(resumo)
display(feature_report)
display(attribute_catalog.head())
display(event_catalog.head())


,n_series,n_classes_globais,n_sources,n_features_originais,n_features_mantidas,n_features_removidas
0,2228,10,3,27,18,9


,column,null_pct,selected_for_modeling,selection_reason,column_type
0,ABER-CKGL,100.0,False,all_null_feature_removed,continuous
1,ABER-CKP,100.0,False,all_null_feature_removed,continuous
2,ESTADO-DHSV,NaN,True,kept_for_modeling,state
3,ESTADO-M1,NaN,True,kept_for_modeling,state
4,ESTADO-M2,NaN,True,kept_for_modeling,state
5,ESTADO-PXO,NaN,True,kept_for_modeling,state
6,ESTADO-SDV-GL,NaN,True,kept_for_modeling,state
7,ESTADO-SDV-P,NaN,True,kept_for_modeling,state
8,ESTADO-W1,NaN,True,kept_for_modeling,state
9,ESTADO-W2,NaN,True,kept_for_modeling,state


,atributo,papel_no_pipeline,descricao_oficial
0,timestamp,metadado,Instant at which observation was generated
1,ABER-CKGL,metadado,Opening of the GLCK (gas lift choke) [%]
2,ABER-CKP,metadado,Opening of the PCK (production choke) [%]
3,ESTADO-DHSV,estado_discreto,"State of the DHSV (downhole safety valve) [0, ..."
4,ESTADO-M1,estado_discreto,"State of the PMV (production master valve) [0,..."


,class_label,event_name,description,transient_event
0,0,NORMAL,Normal Operation,False
1,1,ABRUPT_INCREASE_OF_BSW,Abrupt Increase of BSW,True
2,2,SPURIOUS_CLOSURE_OF_DHSV,Spurious Closure of DHSV,True
3,3,SEVERE_SLUGGING,Severe Slugging,False
4,4,FLOW_INSTABILITY,Flow Instability,False


## Leitura academica

A `versao12` nao e uma ablacao. Ela e uma tentativa de consolidacao. Em termos de desenho experimental, ela pergunta se a combinacao dos melhores blocos das versoes anteriores produz uma representacao temporal mais forte do que cada uma delas separadamente.